In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

AB_IN = "data/GSE280305_paths_AB.h5ad"
S_IN  = "data/GSE280305_gpcca_scalars.h5ad"   # the small scalar-only artifact you ended up with
OUT   = "data/GSE280305_scgeo_lite.h5ad"

# Read AB normally (it has embeddings); read S backed to reduce RAM
adata_ab = sc.read_h5ad(AB_IN)                 # you can switch to backed='r' if needed
adata_s  = sc.read_h5ad(S_IN, backed="r")

common = adata_ab.obs_names.intersection(adata_s.obs_names)
print("common:", len(common), "/", adata_ab.n_obs, "/", adata_s.n_obs)

# --- obs: keep only what we need ---
obs_cols_keep = [c for c in ["timepoint", "sample", "gsm", "leiden_raw"] if c in adata_ab.obs.columns]
obs = adata_ab.obs.loc[common, obs_cols_keep].copy()

# attach scalar columns WITHOUT copying the whole matrix
scalar_cols = [c for c in ["fate_entropy", "fate_margin", "terminal_states", "macrostates"] if c in adata_s.obs.columns]
for c in scalar_cols:
    # robust: pull as numpy from backed object
    obs[c] = np.asarray(adata_s.obs.loc[common, c])

print("attached scalars:", scalar_cols)

# --- obsm: copy only embeddings you want to score/compare ---
obsm = {}
for k in ["X_umap_raw", "X_umap_scanorama", "X_pca_raw", "X_scanorama", "X_pca"]:
    if k in adata_ab.obsm:
        X = adata_ab.obsm[k]
        # store float32 to cut file size in half
        obsm[k] = np.asarray(X, dtype=np.float32)

print("kept obsm:", list(obsm.keys()))

# --- create a tiny AnnData with empty X (0 genes) ---
X_empty = sparse.csr_matrix((len(common), 0), dtype=np.float32)
var = pd.DataFrame(index=pd.Index([], name="gene"))

adata_lite = ad.AnnData(X=X_empty, obs=obs, var=var, obsm=obsm)

# minimal uns: colors are helpful for plots
for uk in ["timepoint_colors", "leiden_raw_colors"]:
    if uk in adata_ab.uns:
        adata_lite.uns[uk] = adata_ab.uns[uk]

adata_lite.write(OUT)
print("Wrote:", OUT, "shape:", adata_lite.shape)


common: 31605 / 31605 / 31605
attached scalars: ['fate_entropy', 'fate_margin']
kept obsm: ['X_umap_raw', 'X_umap_scanorama', 'X_pca_raw', 'X_scanorama', 'X_pca']
Wrote: data/GSE280305_scgeo_lite.h5ad shape: (31605, 0)
